In [4]:
from tiktoken import get_encoding
import torch

In [11]:
token_encoder = get_encoding("gpt2")
example_text = "Your journey starts with one step"

token_ids = token_encoder.encode(example_text)
print(token_ids)

simple_embedder = torch.nn.Embedding(token_encoder.n_vocab, 3)
token_embedded = simple_embedder(torch.tensor(token_ids))
print(token_embedded)
print(token_embedded.shape)

[7120, 7002, 4940, 351, 530, 2239]
tensor([[-2.2268,  1.2862, -1.2914],
        [-0.6693,  0.2088,  0.2676],
        [-1.0527,  0.9033,  0.8143],
        [-0.8627,  0.1313,  0.2572],
        [-0.5692,  0.2280,  0.4910],
        [-0.5114, -1.2017,  1.2100]], grad_fn=<EmbeddingBackward0>)
torch.Size([6, 3])


In [ ]:
attention_scores = token_embedded @ token_embedded.T
print(attention_scores)
attention_weights = attention_scores.softmax(dim=-1)
print(attention_weights)

# Compute context vectors

context_vectors = attention_weights @ token_embedded
print(context_vectors)

tensor([[ 8.2806,  1.4134,  2.4544,  1.7579,  0.9265, -1.9695],
        [ 1.4134,  0.5632,  1.1112,  0.6737,  0.5600,  0.4152],
        [ 2.4544,  1.1112,  2.5873,  1.2363,  1.2050,  0.4382],
        [ 1.7579,  0.6737,  1.2363,  0.8277,  0.6473,  0.5946],
        [ 0.9265,  0.5600,  1.2050,  0.6473,  0.6171,  0.6113],
        [-1.9695,  0.4152,  0.4382,  0.5946,  0.6113,  3.1697]],
       grad_fn=<MmBackward0>)
tensor([[9.9390e-01, 1.0351e-03, 2.9312e-03, 1.4608e-03, 6.3610e-04, 3.5139e-05],
        [2.9085e-01, 1.2429e-01, 2.1498e-01, 1.3881e-01, 1.2389e-01, 1.0719e-01],
        [3.2063e-01, 8.3687e-02, 3.6623e-01, 9.4840e-02, 9.1921e-02, 4.2698e-02],
        [3.3693e-01, 1.1394e-01, 1.9998e-01, 1.3291e-01, 1.1097e-01, 1.0527e-01],
        [1.9106e-01, 1.3243e-01, 2.5241e-01, 1.4451e-01, 1.4021e-01, 1.3940e-01],
        [4.5508e-03, 4.9401e-02, 5.0554e-02, 5.9109e-02, 6.0104e-02, 7.7628e-01]],
       grad_fn=<SoftmaxBackward0>)
tensor([[-2.2186,  1.2815, -1.2801],
        [-1.2023,  0

In [ ]:
attention_weights.sum(axis=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

In [48]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.randn(d_in, d_out))
        self.W_key = nn.Parameter(torch.randn(d_in, d_out))
        self.W_value = nn.Parameter(torch.randn(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        values = x @ self.W_value
        queries = x @ self.W_query

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1] ** 0.5, dim=-1
        )

        context_vectors = attention_weights @ values
        return context_vectors


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        values = self.W_value(x)
        queries = self.W_query(x)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(
            attention_scores / keys.shape[-1] ** 0.5, dim=-1
        )

        context_vectors = attention_weights @ values
        return context_vectors


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(3, 2)
sa_v2 = SelfAttention_v2(3, 2, qkv_bias=False)

x = torch.rand((1, 3))
print(sa_v1(x))
print(sa_v2(x))


tensor([[0.1074, 0.1549]], grad_fn=<MmBackward0>)
tensor([[0.2755, 0.1020]], grad_fn=<MmBackward0>)


In [49]:
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

In [50]:
keys = x @ sa_v1.W_key
values = x @ sa_v1.W_value
queries = x @ sa_v1.W_query

print(x, keys, values, queries)

tensor([[0.4354, 0.0353, 0.1908]]) tensor([[-0.0122,  0.0221]], grad_fn=<MmBackward0>) tensor([[0.2755, 0.1020]], grad_fn=<MmBackward0>) tensor([[-0.2960, -0.0463]], grad_fn=<MmBackward0>)


In [51]:
keys = sa_v2.W_key(x)
values = sa_v2.W_value(x)
queries = sa_v2.W_query(x)

print(x, keys, values, queries)

tensor([[0.4354, 0.0353, 0.1908]]) tensor([[-0.0122,  0.0221]], grad_fn=<MmBackward0>) tensor([[0.2755, 0.1020]], grad_fn=<MmBackward0>) tensor([[-0.2960, -0.0463]], grad_fn=<MmBackward0>)


In [39]:
x = torch.rand((1, 3))
print(sa_v1(x))
print(sa_v2(x))

tensor([[0.0505, 0.0612]], grad_fn=<MmBackward0>)
tensor([[ 0.0851, -0.4065]], grad_fn=<MmBackward0>)


In [41]:
sa_v1.W_query, sa_v1.W_key, sa_v1.W_value

(Parameter containing:
 tensor([[-0.4883,  0.0382],
         [-0.1657, -0.1078],
         [-0.4066, -0.3097]], requires_grad=True),
 Parameter containing:
 tensor([[-0.0455,  0.0908],
         [ 0.0183,  0.5144],
         [-0.0900,  0.3530]], requires_grad=True),
 Parameter containing:
 tensor([[ 0.1361, -0.5366],
         [ 0.2230, -0.3570],
         [-0.0746,  0.4928]], requires_grad=True))

In [44]:
sa_v2.W_query.weight, sa_v2.W_key.weight, sa_v2.W_value.weight

(Parameter containing:
 tensor([[-0.4883, -0.1657, -0.4066],
         [ 0.0382, -0.1078, -0.3097]], requires_grad=True),
 Parameter containing:
 tensor([[-0.0455,  0.0183, -0.0900],
         [ 0.0908,  0.5144,  0.3530]], requires_grad=True),
 Parameter containing:
 tensor([[ 0.1361,  0.2230, -0.0746],
         [-0.5366, -0.3570,  0.4928]], requires_grad=True))